In [ ]:
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

In [ ]:
CRS = 'EPSG:4326'

In [ ]:
# import paths
PATH_TO_ADMIN_BOUNDARIES = '../data/raw/pse_admin_boundaries.shp/pse_admin2.shp'
PATH_TO_3W_COVERAGE = '../data/raw/iati-activity-locations-in-state-of-palestine.csv'

# export paths
PATH_TO_GEOLOCATED_EVENTS_GPKG = '../data/processed/acled/conflict_events_geolocated.gpkg'
PATH_TO_GEOLOCATED_EVENTS_DF = '../data/processed/acled/conflict_events_geolocated.csv'

# import data
df_3w = pd.read_csv(PATH_TO_3W_COVERAGE)
gdf_boundaries = gpd.read_file(PATH_TO_ADMIN_BOUNDARIES)

In [ ]:
df_3w.columns.tolist()

## Geoprocessing

In [ ]:
# convert event coordinates into spatial points
gdf_3w = gpd.GeoDataFrame(
    df_3w,
    geometry=gpd.points_from_xy(
        df_3w['location_longitude'],
        df_3w['location_latitude']
    ),
    crs=CRS
)

In [ ]:
# check and align CRSs
print("Events CRS:", gdf_3w.crs)
print("Admin CRS:", gdf_boundaries.crs)
gdf_3w = gdf_3w.to_crs(gdf_boundaries.crs) # failsafe -- convert events CRS to admin boundaries CRS if not already aligned
assert(gdf_3w.crs == gdf_boundaries.crs)
print('Confirmed CRSs are aligned')

In [ ]:
# perform spatial join (assign event coordinates to an administrative boundary)
gdf_3w_joined = gpd.sjoin(
    gdf_3w,
    gdf_boundaries[['adm1_name', 'adm1_pcode', 'adm2_name', 'adm2_pcode', 'geometry']],
    how='left',
    predicate='within'
)

In [ ]:
print("Total events:", len(gdf_3w_joined))
print("Matched events:", gdf_3w_joined['adm1_name'].notna().sum())
print("Unmatched events:", gdf_3w_joined['adm1_name'].isna().sum())

In [ ]:
unmatched = gdf_3w_joined[gdf_3w_joined['adm1_name'].isna()]
if len(unmatched) > 0:
    display(unmatched.head())
else:
    print('All events matched successfully!')

## Governorate Aggregates

#TODO: IMPORTANT: coverage dates need to be taken into account when doing aggregates (i.e. make sure not to count coverage if the program has terminated)

### Active Org Count

### Sector Coverage

### Population Coverage

Plan:

- figure out how to determine the type of aid (i.e. is this relevant to settler violence? to healthcare? food insecurity?)
- do I need to bring in the non-location dataset to do this?
- aggregate per governorate (active org count, distinct sectors covered, % of population served or per capita coverage)
- build coverage score (org count, sector breadth, per capita coverage, etc) and normalize this on a 0-1 scale
- merge this into a master gdf dataset